# NusantaraLaw: Multi-Model RAG & Latent Space Evaluation
This notebook provides a comprehensive evaluation of the vanilla **Qwen3.5-9B** base model and **4 fine-tuned GGUF models** (Tingkek-1 to Tingkek-4) under two distinct settings:
1. **Without RAG** (Standard Zero-Shot Generation)
2. **With RAG** (Context Retrieval via FAISS using **Qwen3-Embedding-8B** on the test context pool)

### 9 Evaluation Metrics:
- **Lexical:** SacreBLEU, ROUGE-L, METEOR
- **Semantic:** BERTScore (F1), Sentence Similarity, NLI Entailment
- **Generative & Latent:** Perplexity (Fluency), NLaw-Score (Cosine), L2 Latent Distance


## Install Dependencies


In [ ]:
%%capture
# Install dependencies cleanly
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install evaluate rouge_score nltk sacrebleu bert_score sentence-transformers scikit-learn faiss-cpu
!CMAKE_ARGS="-DLLAMA_CUDA=on" pip install llama-cpp-python --upgrade --force-reinstall --no-cache-dir
!pip install penman==1.2.2


## Import Libraries and Setup


In [ ]:
import time
import math
import evaluate
import numpy as np
import nltk
import random
import torch
import torch.nn.functional as F
import os
import json
import pandas as pd
from tqdm import tqdm
from datasets import Dataset
from sentence_transformers import SentenceTransformer, CrossEncoder
from llama_cpp import Llama
from unsloth import FastLanguageModel

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')
nltk.download('omw-1.4')

# Load lexical metrics
bleu_metric      = evaluate.load('sacrebleu')
rouge_metric     = evaluate.load('rouge')
meteor_metric    = evaluate.load('meteor')
bertscore_metric = evaluate.load('bertscore')

# Load semantic models
st_model  = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')
nli_model = CrossEncoder('cross-encoder/nli-deberta-v3-base')

SYSTEM_PROMPT = (
    "Anda adalah seorang pakar hukum Indonesia dan kamus hukum yang sangat presisi. "
    "Tugas Anda adalah memberikan definisi atau penjelasan hukum yang formal, baku, "
    "dan sesuai dengan literatur perundang-undangan. "
    "Jangan merangkum dengan bahasa santai. Gunakan gaya bahasa hukum yang kaku dan tepat."
)
print("Libraries loaded and metric objects initialized successfully!")


## Load Dataset and Sample Evaluation Records


In [ ]:
test_file_path = "/kaggle/input/datasets/adityabayhaqie/nusantara-law-corpus/test-data-reformat.json"

test_data = []
if os.path.exists(test_file_path):
    try:
        with open(test_file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            if isinstance(data, list):
                test_data.extend(data)
                print(f"Successfully loaded {len(data)} test records from: {os.path.basename(test_file_path)}")
    except Exception as e:
        print(f"Error reading {test_file_path}: {e}")
else:
    print(f"Test file not found: {test_file_path}")

# Fix random seed for strict reproducibility
random.seed(3407)
np.random.seed(3407)
torch.manual_seed(3407)

SAMPLE_SIZE = min(20, len(test_data))
eval_indices = random.sample(range(len(test_data)), SAMPLE_SIZE)
eval_samples = [test_data[i] for i in eval_indices]

print(f"Selected {SAMPLE_SIZE} evaluation samples for comparative testing.")


## Load Auxiliary HF Base Model
Used for:
- Text Embeddings extraction (NLaw-Score and L2 distance)
- Perplexity calculations
- Vanilla model zero-shot/RAG generation


In [ ]:
model_name = "unsloth/Qwen3.5-9B"
max_seq_length = 1024
dtype = None
load_in_4bit = True

print(f"Loading HF Base Model: {model_name}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model)
print("HF Base Model loaded successfully.")


## Build FAISS RAG Index
We extract all unique `context` fields from the test dataset and embed them using the powerful 8B parameter embedding model `Qwen/Qwen3-Embedding-8B` (loaded in float16 precision).


In [ ]:
import faiss

print("Initializing Qwen/Qwen3-Embedding-8B in float16...")
embedder = SentenceTransformer(
    'Qwen/Qwen3-Embedding-8B', 
    model_kwargs={'torch_dtype': torch.float16}, 
    trust_remote_code=True
)

# Extract unique context fields from the test pool
unique_contexts = list(set([item.get('context', '').strip() for item in test_data if item.get('context', '').strip()]))
print(f"Found {len(unique_contexts)} unique context chunks in the test pool.")

print("Encoding contexts to build FAISS index...")
context_embeddings = embedder.encode(unique_contexts, show_progress_bar=True)

dimension = context_embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(context_embeddings)
print(f"FAISS index built successfully with {faiss_index.ntotal} contexts.")

def retrieve_contexts(query, k=5):
    query_emb = embedder.encode([query])
    D, I = faiss_index.search(query_emb, k=k)
    retrieved = [unique_contexts[idx] for idx in I[0] if idx != -1]
    return "\n\n".join(retrieved)


## Define Evaluation Helper Functions


In [ ]:
def generate_hf_predictions(samples, rag_contexts=None, max_tokens=512):
    predictions = []
    for idx, sample in enumerate(tqdm(samples, desc="Generating HF Predictions")):
        instruction = sample['instruction']
        context = rag_contexts[idx] if rag_contexts is not None else sample.get('context', '')
        
        user_content = instruction
        if context and context.strip():
            user_content = f"{instruction}\n\nKonteks:\n{context}"
            
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': user_content},
        ]
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        if not prompt_text.endswith('<|im_start|>assistant\n'):
            prompt_text += '<|im_start|>assistant\n'
            
        inputs = tokenizer(text=[prompt_text], return_tensors='pt').to('cuda')
        input_length = inputs['input_ids'].shape[1]
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                use_cache=True,
                do_sample=False,
                repetition_penalty=1.15,
                no_repeat_ngram_size=3
            )
        new_token_ids = outputs[0][input_length:]
        predictions.append(tokenizer.decode(new_token_ids, skip_special_tokens=True).strip())
    return predictions

def generate_gguf_predictions(model_path, samples, rag_contexts=None, max_tokens=512):
    print(f"Loading GGUF model: {os.path.basename(model_path)}...")
    llm = Llama(
        model_path=model_path,
        n_ctx=2048,
        n_gpu_layers=-1,  # Offload all layers to GPU
        verbose=False
    )
    
    predictions = []
    for idx, sample in enumerate(tqdm(samples, desc="Generating GGUF Predictions")):
        instruction = sample['instruction']
        context = rag_contexts[idx] if rag_contexts is not None else sample.get('context', '')
        
        user_content = instruction
        if context and context.strip():
            user_content = f"{instruction}\n\nKonteks:\n{context}"
            
        prompt = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{user_content}<|im_end|>\n<|im_start|>assistant\n"
        
        output = llm(
            prompt,
            max_tokens=max_tokens,
            stop=["<|im_end|>", "<|im_start|>"],
            temperature=0.0,
            repeat_penalty=1.15
        )
        predictions.append(output['choices'][0]['text'].strip())
        
    # Forcefully unload llama_cpp model and free VRAM
    del llm
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(3)
    return predictions

def extract_hidden_vector(text, max_length=512):
    enc = tokenizer(text=text, return_tensors='pt', truncation=True,
                    max_length=max_length, padding=False).to('cuda')
    with torch.no_grad():
        out = model(input_ids=enc['input_ids'], attention_mask=enc['attention_mask'],
                    output_hidden_states=True, return_dict=True)
    last_hidden = out.hidden_states[-1]
    mask = enc['attention_mask'].unsqueeze(-1).float()
    vec = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1)
    return vec.squeeze(0).cpu()

def compute_metrics(preds, refs):
    # 1. Lexical metrics
    preds_rouge = ['\n'.join(nltk.sent_tokenize(p)) for p in preds]
    refs_rouge  = ['\n'.join(nltk.sent_tokenize(r)) for r in refs]
    try: rouge_score = rouge_metric.compute(predictions=preds_rouge, references=refs_rouge, use_stemmer=True)['rougeL'] * 100
    except: rouge_score = 0.0
    try: bleu_score  = bleu_metric.compute(predictions=preds, references=[[r] for r in refs])['score']
    except: bleu_score  = 0.0
    try: meteor_score = meteor_metric.compute(predictions=preds, references=refs)['meteor'] * 100
    except: meteor_score = 0.0
    
    # 2. Semantic metrics
    try: 
        bert_res  = bertscore_metric.compute(predictions=preds, references=refs, lang='id')
        bert_f1   = float(np.mean(bert_res['f1'])) * 100
    except: bert_f1 = 0.0
    try:
        emb_preds = st_model.encode(preds, convert_to_tensor=True)
        emb_refs  = st_model.encode(refs,  convert_to_tensor=True)
        sent_sim  = float(F.cosine_similarity(emb_preds, emb_refs).cpu().mean().item()) * 100
    except: sent_sim = 0.0
    try:
        pairs = [[r, p] for r, p in zip(refs, preds)]
        nli_scores = nli_model.predict(pairs)
        probs      = torch.softmax(torch.tensor(nli_scores), dim=1)
        nli_entail = float(torch.mean(probs[:, 2]).item()) * 100
    except: nli_entail = 0.0
    
    # 3. Perplexity (Fluency)
    total_loss, total_count = 0.0, 0
    for p in preds:
        if not p.strip(): continue
        enc = tokenizer(text=p, return_tensors='pt', truncation=True, max_length=512).to('cuda')
        with torch.no_grad():
            out = model(**enc, labels=enc['input_ids'])
        total_loss  += out.loss.item() * enc['input_ids'].shape[1]
        total_count += enc['input_ids'].shape[1]
    ppl = math.exp(total_loss / total_count) if total_count > 0 else 0.0
    
    # 4. Latent Space Representation metrics
    vecs_ref = []
    vecs_pred = []
    for r, p in zip(refs, preds):
        vecs_ref.append(extract_hidden_vector(r))
        vecs_pred.append(extract_hidden_vector(p))
    vecs_ref  = torch.stack(vecs_ref)
    vecs_pred = torch.stack(vecs_pred)
    
    nlaw_score = float(F.cosine_similarity(vecs_pred, vecs_ref, dim=-1).mean().item()) * 100
    l2_distance = float(torch.norm(vecs_pred - vecs_ref, p=2, dim=-1).mean().item())
    
    return {
        "SacreBLEU": bleu_score,
        "ROUGE-L": rouge_score,
        "METEOR": meteor_score,
        "BERTScore (F1)": bert_f1,
        "Sentence Sim": sent_sim,
        "NLI Entailment": nli_entail,
        "Perplexity": ppl,
        "NLaw Score": nlaw_score,
        "L2 Distance": l2_distance
    }


## Pre-generate RAG Contexts via FAISS
To ensure perfect alignment, we fetch context from the FAISS database for each of the 20 evaluation samples. Immediately afterwards, we delete the 8B embedding model and FAISS index from VRAM to preserve memory for generation.


In [ ]:
rag_contexts = []
for sample in eval_samples:
    rag_contexts.append(retrieve_contexts(sample['instruction'], k=5))
print("RAG Contexts successfully retrieved.")

# Force clean the 8B embedding model and FAISS index to free up VRAM entirely
del embedder
del faiss_index
import gc
gc.collect()
torch.cuda.empty_cache()
print("Qwen3-Embedding-8B model and FAISS index successfully purged from VRAM!")


In [ ]:
gguf_paths = {
    "Tingkek-1": "/kaggle/input/models/adityabayhaqie/limbago/gguf/tingkek-1/1/Qwen3.5-9B.Q4_K_M.gguf",
    "Tingkek-2": "/kaggle/input/models/adityabayhaqie/limbago/gguf/tingkek-2/1/Qwen3.5-9B.Q4_K_M.gguf",
    "Tingkek-3": "/kaggle/input/models/adityabayhaqie/limbago/gguf/tingkek-3/1/Qwen3.5-9B.Q4_K_M.gguf",
    "Tingkek-4": "/kaggle/input/models/adityabayhaqie/limbago/gguf/tingkek-4/1/Qwen3.5-9B.Q4_K_M.gguf"
}

eval_results = {}
all_predictions = {}


## Evaluation: Vanilla — Without RAG


In [ ]:
preds = generate_hf_predictions(eval_samples, rag_contexts=None)
refs = [s['response'] for s in eval_samples]
res = compute_metrics(preds, refs)
eval_results[('Vanilla', 'Without RAG')] = res
all_predictions[('Vanilla', 'Without RAG')] = preds
print("\n--- Results for Vanilla (Without RAG) ---")
for k, v in res.items():
    print(f"  {k:<20}: {v:.4f}")


## Evaluation: Vanilla — With RAG


In [ ]:
preds = generate_hf_predictions(eval_samples, rag_contexts=rag_contexts)
refs = [s['response'] for s in eval_samples]
res = compute_metrics(preds, refs)
eval_results[('Vanilla', 'With RAG')] = res
all_predictions[('Vanilla', 'With RAG')] = preds
print("\n--- Results for Vanilla (With RAG) ---")
for k, v in res.items():
    print(f"  {k:<20}: {v:.4f}")


## Evaluation: Tingkek-1 — Without RAG


In [ ]:
model_path = gguf_paths['Tingkek-1']
preds = generate_gguf_predictions(model_path, eval_samples, rag_contexts=None)
refs = [s['response'] for s in eval_samples]
res = compute_metrics(preds, refs)
eval_results[('Tingkek-1', 'Without RAG')] = res
all_predictions[('Tingkek-1', 'Without RAG')] = preds
print("\n--- Results for Tingkek-1 (Without RAG) ---")
for k, v in res.items():
    print(f"  {k:<20}: {v:.4f}")


## Evaluation: Tingkek-1 — With RAG


In [ ]:
model_path = gguf_paths['Tingkek-1']
preds = generate_gguf_predictions(model_path, eval_samples, rag_contexts=rag_contexts)
refs = [s['response'] for s in eval_samples]
res = compute_metrics(preds, refs)
eval_results[('Tingkek-1', 'With RAG')] = res
all_predictions[('Tingkek-1', 'With RAG')] = preds
print("\n--- Results for Tingkek-1 (With RAG) ---")
for k, v in res.items():
    print(f"  {k:<20}: {v:.4f}")


## Evaluation: Tingkek-2 — Without RAG


In [ ]:
model_path = gguf_paths['Tingkek-2']
preds = generate_gguf_predictions(model_path, eval_samples, rag_contexts=None)
refs = [s['response'] for s in eval_samples]
res = compute_metrics(preds, refs)
eval_results[('Tingkek-2', 'Without RAG')] = res
all_predictions[('Tingkek-2', 'Without RAG')] = preds
print("\n--- Results for Tingkek-2 (Without RAG) ---")
for k, v in res.items():
    print(f"  {k:<20}: {v:.4f}")


## Evaluation: Tingkek-2 — With RAG


In [ ]:
model_path = gguf_paths['Tingkek-2']
preds = generate_gguf_predictions(model_path, eval_samples, rag_contexts=rag_contexts)
refs = [s['response'] for s in eval_samples]
res = compute_metrics(preds, refs)
eval_results[('Tingkek-2', 'With RAG')] = res
all_predictions[('Tingkek-2', 'With RAG')] = preds
print("\n--- Results for Tingkek-2 (With RAG) ---")
for k, v in res.items():
    print(f"  {k:<20}: {v:.4f}")


## Evaluation: Tingkek-3 — Without RAG


In [ ]:
model_path = gguf_paths['Tingkek-3']
preds = generate_gguf_predictions(model_path, eval_samples, rag_contexts=None)
refs = [s['response'] for s in eval_samples]
res = compute_metrics(preds, refs)
eval_results[('Tingkek-3', 'Without RAG')] = res
all_predictions[('Tingkek-3', 'Without RAG')] = preds
print("\n--- Results for Tingkek-3 (Without RAG) ---")
for k, v in res.items():
    print(f"  {k:<20}: {v:.4f}")


## Evaluation: Tingkek-3 — With RAG


In [ ]:
model_path = gguf_paths['Tingkek-3']
preds = generate_gguf_predictions(model_path, eval_samples, rag_contexts=rag_contexts)
refs = [s['response'] for s in eval_samples]
res = compute_metrics(preds, refs)
eval_results[('Tingkek-3', 'With RAG')] = res
all_predictions[('Tingkek-3', 'With RAG')] = preds
print("\n--- Results for Tingkek-3 (With RAG) ---")
for k, v in res.items():
    print(f"  {k:<20}: {v:.4f}")


## Evaluation: Tingkek-4 — Without RAG


In [ ]:
model_path = gguf_paths['Tingkek-4']
preds = generate_gguf_predictions(model_path, eval_samples, rag_contexts=None)
refs = [s['response'] for s in eval_samples]
res = compute_metrics(preds, refs)
eval_results[('Tingkek-4', 'Without RAG')] = res
all_predictions[('Tingkek-4', 'Without RAG')] = preds
print("\n--- Results for Tingkek-4 (Without RAG) ---")
for k, v in res.items():
    print(f"  {k:<20}: {v:.4f}")


## Evaluation: Tingkek-4 — With RAG


In [ ]:
model_path = gguf_paths['Tingkek-4']
preds = generate_gguf_predictions(model_path, eval_samples, rag_contexts=rag_contexts)
refs = [s['response'] for s in eval_samples]
res = compute_metrics(preds, refs)
eval_results[('Tingkek-4', 'With RAG')] = res
all_predictions[('Tingkek-4', 'With RAG')] = preds
print("\n--- Results for Tingkek-4 (With RAG) ---")
for k, v in res.items():
    print(f"  {k:<20}: {v:.4f}")


## Grand Comparative Results Table
Summary of all evaluations.


In [ ]:
rows = []
for key, metrics in eval_results.items():
    row = {
        "Model": key[0],
        "Setting": key[1]
    }
    row.update(metrics)
    rows.append(row)

df_results = pd.DataFrame(rows)
df_results.to_csv("/kaggle/working/multi_model_eval_results.csv", index=False)

# Display formatted comparison
print(df_results.to_markdown(index=False))


## Latent Space Visualizations (t-SNE & PCA)


In [ ]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Extract hidden vectors of reference/ground truth
refs = [s['response'] for s in eval_samples]
vecs_reference = torch.stack([extract_hidden_vector(r) for r in refs]).numpy()

# Extract hidden vectors for each model setting
all_decompositions = {
    "Ground Truth": vecs_reference
}

for (model_name, mode), preds in all_predictions.items():
    # We take Without RAG only for visual clarity, or select specific ones
    lbl = f"{model_name} ({mode})"
    vecs = torch.stack([extract_hidden_vector(p) for p in preds]).numpy()
    all_decompositions[lbl] = vecs

# Combine for t-SNE
combined_vectors = []
labels = []
for lbl, vecs in all_decompositions.items():
    combined_vectors.append(vecs)
    labels.extend([lbl] * len(vecs))
combined_vectors = np.concatenate(combined_vectors, axis=0)

# Perform PCA
pca = PCA(n_components=2, random_state=3407)
pca_res = pca.fit_transform(combined_vectors)

# Perform t-SNE
tsne = TSNE(n_components=2, perplexity=10, random_state=3407)
tsne_res = tsne.fit_transform(combined_vectors)

# Plot PCA
plt.figure(figsize=(15, 6))
plt.subplot(1, 2, 1)
for lbl in all_decompositions.keys():
    indices = [i for i, l in enumerate(labels) if l == lbl]
    plt.scatter(pca_res[indices, 0], pca_res[indices, 1], label=lbl, s=30, alpha=0.7)
plt.title("PCA Representation Space")
plt.legend(fontsize=8)
plt.xlabel("PC 1")
plt.ylabel("PC 2")

# Plot t-SNE
plt.subplot(1, 2, 2)
for lbl in all_decompositions.keys():
    indices = [i for i, l in enumerate(labels) if l == lbl]
    plt.scatter(tsne_res[indices, 0], tsne_res[indices, 1], label=lbl, s=30, alpha=0.7)
plt.title("t-SNE Representation Space")
plt.legend(fontsize=8)
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")

plt.tight_layout()
plt.savefig("/kaggle/working/latent_space_comparison.png", dpi=150, bbox_inches='tight')
plt.show()


## VRAM Purge and Cleanup


In [ ]:
print("Performing final VRAM cleanup...")
try: del model
except: pass
try: del tokenizer
except: pass
import gc
gc.collect()
torch.cuda.empty_cache()
print("Cleanup completed!")
